In [1]:
import pandas as pd

sel = pd.read_parquet("../../data/dataset_v2/selection/sources.parquet")
gold = sel.merge(pd.read_json("../../data/dataset_v2/gold/is_building.jsonl", lines=True), on="source_id")
print("removed:", gold.loc[~gold["is_building"], "assignment"].value_counts().to_dict())
print("final:  ", gold.loc[gold["is_building"], "assignment"].value_counts().to_dict())

removed: {'main': 151, 'ablation': 8}
final:   {'main': 1849, 'ablation': 92}


In [ ]:
# test if empty antrhopic answers still come bck empty
import glob
import hashlib
import json
from pathlib import Path

from thesis.config import REPO_ROOT, load_config
from thesis.vlm.run import configured_clients

prompt_text = next(p["text"] for p in load_config("prompts")["prompts"] if p["id"] == "roleplaying")
prompt_sha = hashlib.sha256(prompt_text.encode()).hexdigest()

empty_images = {
    r["image_sha256"]
    for r in map(
        lambda f: json.loads(Path(f).read_text()), glob.glob(str(REPO_ROOT / "data/vlm_cache/anthropic/*.json"))
    )
    if r["prompt_sha256"] == prompt_sha
    and r["raw_text"].strip() == ""
    and r["complete_request_settings"]["max_tokens"] == 1024
}
images_by_sha = {
    hashlib.sha256(b).hexdigest(): b
    for b in (Path(f).read_bytes() for f in glob.glob(str(REPO_ROOT / "data/ablation/**/*.png"), recursive=True))
}
test_images = [images_by_sha[sha] for sha in sorted(empty_images) if sha in images_by_sha][:50]
len(empty_images), len(test_images)

(675, 50)

In [3]:
client = next(c for c in configured_clients() if c.provider == "anthropic")
results = [client.classify(image, prompt_text).vlm_response for image in test_images]
still_empty = sum(r.raw_text.strip() == "" for r in results)
still_empty, len(results), [r.raw_response.get("stop_reason") for r in results][:10]

(9,
 50,
 ['end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn',
  'end_turn'])